In [1]:
#@title Environment-Aware Installation
import os
import sys
from IPython import get_ipython

# ── Detect environment ────────────────────────────────────────────────
ENV_NAME: str | None = None
BASE_DATA_PATH: str | None = None
BASE_OUTPUT_PATH: str | None = None
DATA_MOUNT: str | None = None
KAGGLE_WHEEL_DIR: str | None = None

os.environ["TORCH_CUDA_ARCH_LIST"] = "12.0"
!export CMAKE_CUDA_ARCHITECTURES="120"
try:
    if "google.colab" in str(get_ipython()):
        ENV_NAME = "colab"
        BASE_DATA_PATH = "/content/"
        BASE_OUTPUT_PATH = "/content/"
        DATA_MOUNT = "/content/ToolFormer/data/generated/v1.0_k5"
        print("Environment: Google Colab")
        print("Will use uv pip install (internet available)")
    elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        ENV_NAME = "kaggle"
        BASE_DATA_PATH = "/kaggle/input/"
        BASE_OUTPUT_PATH = "/kaggle/working/"
        DATA_MOUNT = "/kaggle/input/datasets/dzung271828/toolformer-data/generated/v1.0_k5"
        KAGGLE_WHEEL_DIR = "/kaggle/input/datasets/dzung271828/telco-wheels/telco-wheels/telco-wheels"
        print("Environment: Kaggle")
        print("Will use pip --no-index --find-links (offline mode)")
        os.environ["HF_DATASETS_OFFLINE"] = "1"
        os.environ["TRANSFORMERS_OFFLINE"] = "1"
        os.environ["HF_HUB_OFFLINE"] = "1"
    else:
        ENV_NAME = "local"
        BASE_DATA_PATH = "./data/"
        BASE_OUTPUT_PATH = "./output/"
        DATA_MOUNT = "data/generated/v1.0_k5"
        print("Environment: Local")
except NameError:
    ENV_NAME = "local"
    BASE_DATA_PATH = "./data/"
    BASE_OUTPUT_PATH = "./output/"
    DATA_MOUNT = "data/generated/v1.0_k5"
    print("Non-interactive session. Using local paths.")

os.makedirs(BASE_OUTPUT_PATH, exist_ok=True)
print(f"Environment: {ENV_NAME}")
print(f"Base data path: {BASE_DATA_PATH}")
print(f"Base output path: {BASE_OUTPUT_PATH}")
print(f"Data mount: {DATA_MOUNT}")
if KAGGLE_WHEEL_DIR:
    print(f"Kaggle wheel dir: {KAGGLE_WHEEL_DIR}")

# ── Detect GPU type ──────────────────────────────────────────────────
import subprocess as _sp
IS_T4_GPU = False
try:
    _gpu_name = (
        _sp.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            text=True,
        )
        .strip()
        .split("\n")[0]
    )
    IS_T4_GPU = "Tesla T4" in _gpu_name
    print(f"GPU type: {_gpu_name} → {'T4 (internet install)' if IS_T4_GPU else 'non-T4 (offline install)'}")
except Exception:
    print("GPU detection: nvidia-smi unavailable → defaulting to offline install")
print(f"IS_T4_GPU: {IS_T4_GPU}")

Environment: Kaggle
Will use pip --no-index --find-links (offline mode)
Environment: kaggle
Base data path: /kaggle/input/
Base output path: /kaggle/working/
Data mount: /kaggle/input/datasets/dzung271828/toolformer-data/generated/v1.0_k5
Kaggle wheel dir: /kaggle/input/datasets/dzung271828/telco-wheels/telco-wheels/telco-wheels
GPU type: NVIDIA RTX PRO 6000 Blackwell Server Edition → non-T4 (offline install)
IS_T4_GPU: False


In [2]:
# ── Install packages ────────────────────────────────────────────────
os.environ["UNSLOTH_VLLM_STANDBY"] = "0"

if ENV_NAME == "colab":
    print("Installing packages for colab env...")
    !pip install --upgrade -qqq uv
    try:
        import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except:
        _numpy = "numpy"; _pil = "pillow"
    try:
        import subprocess
        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except:
        is_t4 = False
    _vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
    !uv pip install transformers==4.56.2
    !uv pip install --no-deps trl==0.22.2
elif ENV_NAME == "kaggle":
    print("Installing packages for kaggle env...")
    import subprocess
    subprocess.run(
        "pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages "
        "unsloth trl peft transformers datasets accelerate bitsandbytes vllm",
        shell=True, check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True, check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True, check=True,
    )
    for _wd in ["/kaggle/input/datasets/llkh0a/rtx-wheels/wheels"]:
        if os.path.isdir(_wd):
            subprocess.run(
                ["pip", "install", "-q", "--no-index", "--find-links", _wd,
                 "protobuf==6.33.5", "sentencepiece", "safetensors", "huggingface_hub", "vllm"],
                check=False,
            )
    subprocess.run("rm -rf /kaggle/tmp/*", shell=True, check=True)
else:
    print("Installing packages for local env...")
    !pip install unsloth vllm
    !pip install transformers==4.56.2
    !pip install trl==0.22.2

if ENV_NAME == "kaggle":
    !pip install --no-index --find-links=/kaggle/input/datasets/nctuan/nvidia-offline-packages-nemotron/ /kaggle/input/datasets/nctuan/nvidia-offline-packages-nemotron/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

print("Core stack installation complete.")

Installing packages for kaggle env...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
google-adk 1.25.1 requires opentelemetry-api<1.40.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.25.1 requires opentelemetry-sdk<1.40.0,>=1.36.0, but you have opentelem

Looking in links: /kaggle/input/datasets/nctuan/nvidia-offline-packages-nemotron/
Processing /kaggle/input/datasets/nctuan/nvidia-offline-packages-nemotron/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Core stack installation complete.


In [3]:
#@title Check installed library versions
import sys
import importlib

_LIBS = {
    "unsloth": None, "unsloth_zoo": None, "torch": None,
    "transformers": None, "trl": None, "peft": None,
    "vllm": None, "datasets": None, "accelerate": None,
    "bitsandbytes": None, "flash_attn": None,
}

print(f"{'Library':<20} {'Version':<20}")
print("-" * 40)
for lib_name in _LIBS:
    try:
        mod = importlib.import_module(lib_name)
        ver = getattr(mod, "__version__", "no __version__")
        print(f"{lib_name:<20} {ver:<20}")
    except ImportError:
        print(f"{lib_name:<20} {'not installed':<20}")

Library              Version             
----------------------------------------
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-08-22 11:06:49.146634: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787396809.316178      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787396809.365094      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787396809.776973      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787396809.776984      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787396809.776985      65 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth              2026.3.17           
unsloth_zoo          2026.3.6            
torch                2.10.0+cu128        
transformers         4.57.6              
trl                  0.24.0              
peft                 0.18.1              
vllm                 0.18.0              
datasets             4.3.0               
accelerate           1.12.0              
bitsandbytes         0.49.2              
flash_attn           2.8.3               


In [4]:
import os
import sys
from IPython import get_ipython


def load_secret(key_name: str) -> str | None:
    """Load a secret from environment-specific secret stores.

    Args:
        key_name: Name of the secret key to load.

    Returns:
        The secret value if found, otherwise None.
    """
    env = ENV_NAME
    secret_value = None
    print(f"Attempting to load secret '{key_name}' from '{env}' environment...")
    try:
        if env == "colab":
            from google.colab import userdata
            secret_value = userdata.get(key_name)
        elif env == "kaggle":
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            secret_value = user_secrets.get_secret(key_name)
        else:
            secret_value = os.getenv(key_name)
        if not secret_value:
            print(f"Secret '{key_name}' not found in the {env} environment.")
            return None
        print(f"Successfully loaded secret '{key_name}'.")
        return secret_value
    except Exception as e:
        print(f"An error occurred while loading secret '{key_name}': {e}")
        return None


def print_system_info():
    """Print Python version, PyTorch/CUDA info, GPU count and nvidia-smi output."""
    print("\n🔧 System Information")
    print(f"Python version: {sys.version.split()[0]}")
    try:
        import torch
        print(f"PyTorch version: {torch.__version__}")
        if torch.cuda.is_available():
            print(f"CUDA version: {torch.version.cuda}")
            print(f"GPU count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("CUDA not available")
    except ImportError:
        print("PyTorch not installed")
    finally:
        !nvidia-smi


is_kaggle = ENV_NAME == "kaggle"
is_colab = ENV_NAME == "colab"
is_local = ENV_NAME == "local"
print_system_info()

if not is_kaggle:
    os.environ["WANDB_API_KEY"] = wandb_key = load_secret("WANDB_API_KEY")
    os.environ["HF_TOKEN"] = HF_TOKEN = load_secret("HF_TOKEN")
    GITHUB_TOKEN = load_secret("GITHUB_TOKEN")


🔧 System Information
Python version: 3.12.12
PyTorch version: 2.10.0+cu128
CUDA version: 12.8
GPU count: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition
Sat Aug 22 11:07:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   36C    P0             87W /  600W |     681MiB /  97887MiB |  

In [5]:
!find /usr -name "libcuda.so*" 2>/dev/null

/usr/local/nvidia/lib64/libcuda.so.580.159.04
/usr/local/nvidia/lib64/libcuda.so.1
/usr/local/nvidia/lib64/libcuda.so
/usr/local/cuda-12.8/compat/libcuda.so.1
/usr/local/cuda-12.8/compat/libcuda.so
/usr/local/cuda-12.8/compat/libcuda.so.570.124.06


In [6]:
import os

nvidia_lib_dir = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = nvidia_lib_dir + ":" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = nvidia_lib_dir + ":" + os.environ.get("LD_LIBRARY_PATH", "")

print("Environment paths for libcuda successfully configured!")
print(os.environ["LIBRARY_PATH"])
print(os.environ["LD_LIBRARY_PATH"])
!rm -rf ~/.cache/torch_extensions/

Environment paths for libcuda successfully configured!
/usr/local/nvidia/lib64:/usr/local/cuda/lib64/stubs
/usr/local/nvidia/lib64:/usr/local/lib/python3.12/dist-packages/cv2/../../lib64:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/cuda/lib64


### Unsloth
#
Goal: To convert `Qwen3-4B-Base` into a reasoning model via GRPO by using OpenR1's Math dataset.
#
We first pre fine-tune the model to make GRPO skip trying to match formatting - this speeds GRPO up.

In [7]:
#@title === Centralized Configuration & Utilities ===
import argparse
import gc
import json
import logging
import os
import pathlib
import random
import sys
from dataclasses import dataclass, field, asdict
from typing import Any, Optional, Tuple

import numpy as np
import torch

# ── Logging setup ─────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("grpo_pipeline")


# ── Constants ─────────────────────────────────────────────────────────
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
DEFAULT_LORA_RANK = 32
GPU_MEMORY_UTILIZATION = 0.6
SFT_SAVE_DIR = "qwen_lora_sft"
GRPO_LORA_SAVE_NAME = "grpo_saved_lora"

REASONING_START = "<start_working_out>"
REASONING_END = "<end_working_out>"
SOLUTION_START = "<SOLUTION>"
SOLUTION_END = "</SOLUTION>"

SYSTEM_PROMPT = (
    f"You are given a problem.\n"
    f"Think about the problem and provide your working out.\n"
    f"Place it between {REASONING_START} and {REASONING_END}.\n"
    f"Then, provide your solution between {SOLUTION_START}{SOLUTION_END}"
)


# ── ExperimentConfig dataclass ────────────────────────────────────────
@dataclass
class ExperimentConfig:
    """Centralized configuration for the GRPO math-reasoning training pipeline.

    Encapsulates all hyperparameters, data paths, LoRA settings, ablation knobs,
    and tracking options for both SFT (Stage 1) and GRPO (Stage 2) phases.

    Usage:
        # From CLI (argparse)
        cfg = ExperimentConfig.from_argparse(parse_args())

        # Directly
        cfg = ExperimentConfig(model_path="...", max_seq_length=8192, ...)

    Attributes:
        model_path: HF model id or local path to the base model.
        grpo_data_path: Path to the GRPO training dataset.
        sft_data_path: Path to the pre-SFT formatting dataset.
        adapter_path: If set, load this LoRA adapter into the base model
            (used for GRPO Stage 2 warm-start from a saved SFT adapter).
            If None, a fresh LoRA adapter is created via ``get_peft_model``.
        sft_save_dir: Directory where the SFT adapter is saved.
        sample_n: Subset GRPO data to first N samples (0 = all).
        max_seq_length: Maximum sequence length (context window).
        sft_epochs: Number of pre-SFT epochs.
        no_sft: If True, skip pre-SFT (GRPO-from-base comparison arm).
        lora_rank: LoRA rank (r). Alpha is set to 2 * rank.
        beta: KL-divergence penalty coefficient (0 disables KL).
        num_generations: Group size — completions sampled per prompt.
        importance_sampling_level: Advantage credit granularity ("token"|"sequence").
        loss_type: GRPO loss variant ("grpo"|"bnpo"|"dr_grpo"|"dapo"|"cispo").
        temperature: Sampling temperature for generation.
        lr: Learning rate.
        optim: Optimizer name.
        max_steps: Maximum training steps.
        seed: Global random seed.
        run_name: Run identifier (auto-generated if None).
        wandb: Enable Weights & Biases logging.
        wandb_project: W&B project name.
        wandb_group: W&B run group.
        offline_mode: Force W&B offline mode.
        output_dir: Computed output directory (set in __post_init__).
    """

    # --- Model / data paths ---
    model_path: str = "/kaggle/input/models/qwen-lm/qwen-3/transformers/1.7b-base/1"
    grpo_data_path: str = "/kaggle/input/datasets/alejopaullier/openr1-math-220k"
    sft_data_path: str = "/kaggle/input/datasets/ga5534/openmathreasoning-cot-kaggle"
    adapter_path: Optional[str] = None
    sft_save_dir: str = SFT_SAVE_DIR

    # --- Sequence / sampling ---
    sample_n: int = 0
    max_seq_length: int = 16384
    sft_epochs: int = 1
    no_sft: bool = True

    # --- LoRA ---
    lora_rank: int = DEFAULT_LORA_RANK

    # --- GRPO ablation knobs ---
    beta: float = 0.0
    num_generations: int = 8
    importance_sampling_level: str = "token"
    loss_type: str = "dapo"
    temperature: float = 1.0
    lr: float = 5e-5
    optim: str = "adamw_torch"
    max_steps: int = 500
    seed: int = 3407

    # --- Run identity / tracking ---
    run_name: Optional[str] = None
    wandb: bool = False
    wandb_project: str = "grpo-math-ablation"
    wandb_group: str = "default"
    offline_mode: bool = True

    # --- Derived (computed in __post_init__) ---
    output_dir: Optional[pathlib.Path] = field(default=None, init=False)

    def __post_init__(self) -> None:
        """Compute derived fields after dataclass initialization."""
        if self.run_name is None:
            self.run_name = self._generate_run_name()
        self.output_dir = pathlib.Path("outputs") / self.run_name
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def _generate_run_name(self) -> str:
        """Auto-generate a run name from ablation axes."""
        parts = [
            f"beta{self.beta}",
            f"ng{self.num_generations}",
            f"loss_{self.loss_type}",
            f"is_{self.importance_sampling_level}",
            f"temp{self.temperature}",
            f"lr{self.lr}",
            f"opt{self.optim}",
        ]
        if self.no_sft:
            parts.append("noSFT")
        if self.adapter_path:
            parts.append("stage2")
        if self.sample_n:
            parts.append(f"n{self.sample_n}")
        return "_".join(parts)

    def to_dict(self) -> dict:
        """Serialize config to a JSON-safe dictionary."""
        d = asdict(self)
        if d.get("output_dir") is not None:
            d["output_dir"] = str(d["output_dir"])
        return d

    @classmethod
    def from_argparse(cls, ns: argparse.Namespace) -> "ExperimentConfig":
        """Create an ExperimentConfig from an argparse.Namespace."""
        return cls(
            model_path=ns.model_path,
            grpo_data_path=ns.grpo_data_path,
            sft_data_path=ns.sft_data_path,
            adapter_path=getattr(ns, "adapter_path", None),
            sample_n=ns.sample_n,
            max_seq_length=ns.max_seq_length,
            sft_epochs=ns.sft_epochs,
            no_sft=ns.no_sft,
            lora_rank=getattr(ns, "lora_rank", DEFAULT_LORA_RANK),
            beta=ns.beta,
            num_generations=ns.num_generations,
            importance_sampling_level=ns.importance_sampling_level,
            loss_type=ns.loss_type,
            temperature=ns.temperature,
            lr=ns.lr,
            optim=ns.optim,
            max_steps=ns.max_steps,
            seed=ns.seed,
            run_name=ns.run_name,
            wandb=ns.wandb,
            wandb_project=ns.wandb_project,
            wandb_group=ns.wandb_group,
            offline_mode=ns.offline_mode,
        )


# ── Argument parser ───────────────────────────────────────────────────
def parse_args(argv=None) -> argparse.Namespace:
    """Parse CLI arguments for the GRPO training pipeline.

    All defaults mirror ExperimentConfig defaults so that the two stay
    in sync.  Use ``ExperimentConfig.from_argparse(ns)`` to obtain the
    dataclass.
    """
    p = argparse.ArgumentParser(
        description="Qwen3-4B GRPO math-reasoning fine-tuning (ablation-ready, WandB-tracked).",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )

    # --- Model / data paths ---
    p.add_argument("--model-path", type=str,
                   default="/kaggle/input/models/qwen-lm/qwen-3/transformers/1.7b-base/1",
                   help="HF model id or local path (offline: /kaggle/input/...)")
    p.add_argument("--grpo-data-path", type=str,
                   default="/kaggle/input/datasets/alejopaullier/openr1-math-220k",
                   help="GRPO training dataset")
    p.add_argument("--sft-data-path", type=str,
                   default="/kaggle/input/datasets/ga5534/openmathreasoning-cot-kaggle",
                   help="Pre-SFT formatting dataset")
    p.add_argument("--adapter-path", type=str, default=None,
                   help="Path to a saved LoRA adapter to warm-start GRPO Stage 2. "
                        "If None, a fresh LoRA adapter is created.")
    p.add_argument("--lora-rank", type=int, default=DEFAULT_LORA_RANK,
                   help="LoRA rank (r). Alpha = 2 * rank.")
    p.add_argument("--sample-n", type=int, default=0,
                   help="Subset to first N GRPO samples (0 = use all)")
    p.add_argument("--max-seq-length", type=int, default=16384,
                   help="Max sequence length (context window)")
    p.add_argument("--sft-epochs", type=int, default=1, help="Pre-SFT epochs")
    p.add_argument("--no-sft", default=True, action="store_true",
                   help="Skip pre-SFT formatting (GRPO-from-base comparison arm)")

    # --- GRPO ablation knobs ---
    p.add_argument("--beta", type=float, default=0.0,
                   help="KL-divergence penalty coefficient (0 disables KL & its logging)")
    p.add_argument("--num-generations", type=int, default=8, choices=[2, 4, 8, 16],
                   help="Group size: completions sampled per prompt")
    p.add_argument("--importance-sampling-level", type=str, default="token",
                   choices=["token", "sequence"],
                   help="Advantage credit assignment granularity")
    p.add_argument("--loss-type", type=str, default="cispo",
                   choices=["grpo", "bnpo", "dr_grpo", "dapo", "cispo"],
                   help="GRPO loss variant")
    p.add_argument("--temperature", type=float, default=1.0, help="Sampling temperature")
    p.add_argument("--lr", type=float, default=5e-5, help="Learning rate")
    p.add_argument("--optim", type=str, default="adamw_torch",
                   choices=["adamw_torch", "adamw_8bit"],
                   help="Optimizer")
    p.add_argument("--max-steps", type=int, default=250, help="Training steps")
    p.add_argument("--seed", type=int, default=3407, help="Global seed")

    # --- Run identity / tracking ---
    p.add_argument("--run-name", type=str, default=None,
                   help="Run id (auto-generated from ablation axes if omitted)")
    p.add_argument("--wandb", dest="wandb", action="store_true", default=False,
                   help="Enable WandB logging")
    p.add_argument("--no-wandb", dest="wandb", action="store_false",
                   help="Disable WandB logging")
    p.add_argument("--wandb-project", type=str, default="grpo-math-ablation",
                   help="WandB project (env WANDB_PROJECT overrides)")
    p.add_argument("--wandb-group", type=str, default="default",
                   help="WandB run group")
    p.add_argument("--offline-mode", action="store_true", default=True,
                   help="Force WandB offline (no network; sync later)")

    return p.parse_args(args=["--offline-mode"])


# ── Build config ───────────────────────────────────────────────────────
_args = parse_args()
config = ExperimentConfig.from_argparse(_args)
RUN_NAME = config.run_name
output_dir = config.output_dir

logger.info("ExperimentConfig created:")
logger.info(f"  run_name       = {config.run_name}")
logger.info(f"  output_dir     = {config.output_dir}")
logger.info(f"  model_path     = {config.model_path}")
logger.info(f"  adapter_path   = {config.adapter_path or '(none — fresh LoRA)'}")
logger.info(f"  lora_rank      = {config.lora_rank}")
logger.info(f"  no_sft         = {config.no_sft}")
logger.info(f"  loss_type      = {config.loss_type}")
logger.info(f"  num_generations= {config.num_generations}")
logger.info(f"  beta           = {config.beta}")


# ── WandB env setup ───────────────────────────────────────────────────
if config.wandb:
    os.environ.setdefault("WANDB_API_KEY", os.environ.get("WANDB_API_KEY", ""))
    os.environ.setdefault("WANDB_PROJECT", config.wandb_project)
    os.environ.setdefault("WANDB_RUN_GROUP", config.wandb_group)
    os.environ.setdefault("WANDB_NAME", RUN_NAME)
    if config.offline_mode:
        os.environ["WANDB_MODE"] = "offline"
        logger.info("[wandb] Offline mode enabled — sync later with `wandb sync`")
    if not os.environ.get("WANDB_API_KEY"):
        logger.warning("[wandb] WANDB_API_KEY not found — falling back to offline mode")
        os.environ["WANDB_MODE"] = "offline"


# ── Utility functions ─────────────────────────────────────────────────
def set_seed(seed: int) -> None:
    """Set all random seeds for reproducibility.

    Args:
        seed: Integer seed for Python ``random``, ``numpy``, and ``torch``.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    logger.info(f"Global seed set to {seed}")


def cleanup_gpu() -> None:
    """Force garbage collection and release GPU cache between training stages."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    logger.info("GPU memory cleared (gc.collect + empty_cache)")


def save_config_and_hyperparams(
    cfg: ExperimentConfig,
    tokenizer=None,
    extra: Optional[dict] = None,
) -> dict:
    """Persist full config and a concise hyperparams summary to disk.

    Writes two JSON files into ``cfg.output_dir``:
      * ``config.json``      — the full ExperimentConfig (all fields).
      * ``hyperparams.json``  — a curated subset of the most important knobs.

    If W&B is enabled, also pushes the hyperparams dict to the active run.

    Args:
        cfg: The ExperimentConfig dataclass.
        tokenizer: Optional tokenizer (unused in serialization but kept
            for API compatibility).
        extra: Additional key-value pairs to merge into hyperparams.

    Returns:
        The hyperparams dictionary that was saved.
    """
    output_dir = pathlib.Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Save full config → config.json
    config_path = output_dir / "config.json"
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(cfg.to_dict(), f, indent=2, ensure_ascii=False, default=str)

    # 2. Save curated hyperparams → hyperparams.json
    hyperparams = {
        "model_path": cfg.model_path,
        "adapter_path": cfg.adapter_path,
        "max_seq_length": cfg.max_seq_length,
        "lora_rank": cfg.lora_rank,
        "sft_epochs": cfg.sft_epochs,
        "learning_rate": cfg.lr,
        "beta": cfg.beta,
        "num_generations": cfg.num_generations,
        "loss_type": cfg.loss_type,
        "importance_sampling_level": cfg.importance_sampling_level,
        "temperature": cfg.temperature,
        "optim": cfg.optim,
        "max_steps": cfg.max_steps,
        "seed": cfg.seed,
        "run_name": cfg.run_name,
    }
    if extra:
        hyperparams.update(extra)

    hparams_path = output_dir / "hyperparams.json"
    with open(hparams_path, "w", encoding="utf-8") as f:
        json.dump(hyperparams, f, indent=2, ensure_ascii=False, default=str)

    logger.info(f"[tracking] Saved config to {config_path}")
    logger.info(f"[tracking] Saved hyperparams to {hparams_path}")

    # 3. If wandb is active, update its config
    if cfg.wandb:
        try:
            import wandb
            wandb.config.update(hyperparams, allow_val_change=True)
        except Exception:
            pass

    return hyperparams


# ── Model loading ─────────────────────────────────────────────────────
def load_model(
    config: ExperimentConfig,
    mode: str = "train",
    stage: str = "sft",
    adapter_path: Optional[str] = None,
) -> Tuple[Any, Any]:
    """Load a base model, optionally attaching a pre-trained LoRA adapter.

    This is the single entry-point for model construction across all
    pipeline phases (SFT, GRPO, eval).  It:

    1. Sets the ``UNSLOTH_VLLM_STANDBY`` env var and ``fast_inference``
       flag according to ``(mode, stage)``.
    2. Loads the base model + tokenizer via ``FastLanguageModel``.
    3. **If ``adapter_path`` is not None**: loads the saved LoRA adapter
       into the base model using ``peft.PeftModel.from_pretrained``.
       This is the GRPO Stage 2 warm-start path — the adapter's own
       ``adapter_config.json`` determines rank, target modules, etc.
    4. **If ``adapter_path`` is None**: creates a fresh LoRA adapter
       via ``FastLanguageModel.get_peft_model``.

    Args:
        config: ExperimentConfig with model path, LoRA rank, seed, etc.
        mode: "train" or "eval".
        stage: "sft", "grpo", or "eval".
        adapter_path: Override for ``config.adapter_path``.  If provided
            (non-None), the adapter is loaded into the base model.
            If None and ``config.adapter_path`` is set, that value is
            used.  If both are None, a fresh LoRA is created.

    Returns:
        Tuple of (model, tokenizer).
    """
    assert mode in ("train", "eval"), f"mode must be 'train' or 'eval', got '{mode}'"
    assert stage in ("sft", "grpo", "eval"), f"stage must be 'sft' | 'grpo' | 'eval', got '{stage}'"

    # Resolve adapter_path: explicit parameter takes precedence over config
    resolved_adapter = adapter_path if adapter_path is not None else config.adapter_path

    # ── Determine vLLM standby & fast_inference flags ────────────────
    is_sft_train = (mode == "train" and stage == "sft")
    is_grpo_train = (mode == "train" and stage == "grpo")

    if is_sft_train:
        os.environ["UNSLOTH_VLLM_STANDBY"] = "0"
        fast_inference_flag = False
        logger.info("[load_model] MODE: TRAIN-SFT → UNSLOTH_VLLM_STANDBY=0, fast_inference=False")
    elif is_grpo_train:
        os.environ["UNSLOTH_VLLM_STANDBY"] = "0"
        fast_inference_flag = True
        logger.info("[load_model] MODE: TRAIN-GRPO → UNSLOTH_VLLM_STANDBY=0, fast_inference=True")
    else:  # eval
        os.environ["UNSLOTH_VLLM_STANDBY"] = "0"
        fast_inference_flag = True
        logger.info(f"[load_model] MODE: {mode.upper()}-{stage.upper()} → fast_inference=True")

    # ── Load base model (import AFTER env setup — critical for Unsloth) ──
    from unsloth import FastLanguageModel

    logger.info(f"[load_model] Loading base model from: {config.model_path}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.model_path,
        max_seq_length=config.max_seq_length,
        load_in_4bit=False,
        fast_inference=fast_inference_flag,
        max_lora_rank=config.lora_rank,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        attn_implementation="flash_attention_2",
        dtype=torch.bfloat16,
    )

    # ── Attach LoRA adapter ──────────────────────────────────────────
    if resolved_adapter is not None:
        # ── Load existing adapter (GRPO Stage 2 warm-start) ──────────
        from peft import PeftModel

        is_trainable = (mode == "train")
        logger.info(
            f"[load_model] Loading LoRA adapter from: {resolved_adapter} "
            f"(is_trainable={is_trainable})"
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=config.lora_rank,
            target_modules=LORA_TARGET_MODULES,
            lora_alpha=config.lora_rank * 2,
            use_gradient_checkpointing="unsloth",
            random_state=config.seed,
        )
        model.load_adapter(resolved_adapter, adapter_name="default")
        
        # Enable gradient checkpointing for training efficiency
        if is_trainable:
            model.gradient_checkpointing_enable()
            if hasattr(model, "enable_input_require_grads"):
                model.enable_input_require_grads()

        logger.info(
            f"[load_model] Adapter loaded successfully. "
            f"Active adapters: {list(model.peft_config.keys()) if hasattr(model, 'peft_config') else 'N/A'}"
        )
    else:
        # ── Create fresh LoRA adapter ────────────────────────────────
        logger.info(
            f"[load_model] Creating fresh LoRA adapter "
            f"(r={config.lora_rank}, alpha={config.lora_rank * 2})"
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=config.lora_rank,
            target_modules=LORA_TARGET_MODULES,
            lora_alpha=config.lora_rank * 2,
            use_gradient_checkpointing="unsloth",
            random_state=config.seed,
        )
        logger.info("[load_model] Fresh LoRA adapter created.")

    return model, tokenizer


# ── Set global seed ───────────────────────────────────────────────────
set_seed(config.seed)

### GRPO chat template
Since we're using a base model, we should set a chat template.

In [8]:
# ── Load model for chat-template setup (SFT settings) ────────────────
# We load once here so the tokenizer is available for chat-template
# configuration and dataset tokenisation.  If SFT is enabled, this same
# model is reused; otherwise it is freed before GRPO.
model, tokenizer = load_model(
    config=config,
    mode="train",
    stage="sft",
)

# ── Build chat template ───────────────────────────────────────────────
chat_template = (
    "{% if messages[0]['role'] == 'system' %}"
    "{{ messages[0]['content'] + eos_token }}"
    "{% set loop_messages = messages[1:] %}"
    "{% else %}"
    "{{ '{system_prompt}' + eos_token }}"
    "{% set loop_messages = messages %}"
    "{% endif %}"
    "{% for message in loop_messages %}"
    "{% if message['role'] == 'user' %}"
    "{{ message['content'] }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ message['content'] + eos_token }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"
    "{% endif %}"
)

chat_template = chat_template.replace(
    "'{system_prompt}'", f"'{SYSTEM_PROMPT}'"
).replace("'{reasoning_start}'", f"'{REASONING_START}'")
tokenizer.chat_template = chat_template

==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [9]:
# Verify chat template
tokenizer.apply_chat_template(
    [
        {"role": "user", "content": "What is 1+1?"},
        {
            "role": "assistant",
            "content": f"{REASONING_START}I think it's 2.{REASONING_END}{SOLUTION_START}2{SOLUTION_END}",
        },
        {"role": "user", "content": "What is 2+2?"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION><|im_end|>What is 1+1?<start_working_out>I think it's 2.<end_working_out><SOLUTION>2</SOLUTION><|im_end|>What is 2+2?<start_working_out>"

### Pre fine-tuning for formatting

In [10]:
from functools import partial


def process_batch(
    batch: dict,
    tokenizer,
    reasoning_start: str,
    reasoning_end: str,
    solution_start: str,
    solution_end: str,
    system_prompt: str,
) -> dict:
    """Tokenise a batch of SFT examples into chat-formatted text.

    Args:
        batch: A dict of lists with keys "problem", "generated_solution",
            "expected_answer".
        tokenizer: HuggingFace tokenizer with a chat template.
        reasoning_start / reasoning_end: Tags wrapping the chain-of-thought.
        solution_start / solution_end: Tags wrapping the final answer.
        system_prompt: The system prompt string.

    Returns:
        Dict with keys "text" (decoded strings) and "n_tokens" (int lengths).
    """
    messages_list = []
    for i in range(len(batch["problem"])):
        thoughts = (
            batch["generated_solution"][i]
            .replace("ulla", "")
            .replace("енного", "")
            .strip()
        )
        final_prompt = (
            reasoning_start + thoughts + reasoning_end
            + solution_start + batch["expected_answer"][i] + solution_end
        )
        messages_list.append([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": batch["problem"][i]},
            {"role": "assistant", "content": final_prompt},
        ])
    tokenized_batch = tokenizer.apply_chat_template(messages_list, tokenize=True)
    texts = [tokenizer.decode(ids) for ids in tokenized_batch]
    n_tokens = [len(ids) for ids in tokenized_batch]
    return {"text": texts, "n_tokens": n_tokens}


# ── SFT training block ────────────────────────────────────────────────
if not config.no_sft:
    logger.info("[SFT] Pre-SFT formatting fine-tune enabled")

    # Save config before training
    save_config_and_hyperparams(config, tokenizer, extra={"phase": "sft"})

    # ── Load dataset & filter ───────────────────────────────────────
    from datasets import load_dataset

    logger.info("[SFT] Loading dataset...")
    dataset = load_dataset(config.sft_data_path, "default", split="train")
    logger.info(f"[SFT] Raw size: {len(dataset)}")

    def is_numeric(example: dict) -> bool:
        try:
            float(example["expected_answer"])
            return True
        except (ValueError, TypeError):
            return False

    dataset = dataset.filter(is_numeric)
    logger.info(f"[SFT] After numeric filter: {len(dataset)}")

    dataset = dataset.map(
        process_batch,
        batched=True,
        batch_size=1000,
        remove_columns=dataset.column_names,
        num_proc=os.cpu_count(),
        fn_kwargs={
            "tokenizer": tokenizer,
            "reasoning_start": REASONING_START,
            "reasoning_end": REASONING_END,
            "solution_start": SOLUTION_START,
            "solution_end": SOLUTION_END,
            "system_prompt": SYSTEM_PROMPT,
        },
    )
    logger.info(f"[SFT] After processing: {len(dataset)}")
    dataset = dataset.filter(lambda x: x["n_tokens"] <= config.max_seq_length // 5)

    logger.info(f"[SFT] Data shape: {dataset.shape}")
    logger.info(f"[SFT] Sample text:\n{dataset['text'][0][:500]}...")

    # ── SFT Trainer ─────────────────────────────────────────────────
    from trl import SFTTrainer, SFTConfig

    logger.info("[SFT] Starting SFT training...")
    sft_config = SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=16,
        gradient_accumulation_steps=1,
        warmup_ratio=0.1,
        num_train_epochs=config.sft_epochs,
        learning_rate=2e-4,
        logging_steps=5,
        optim=config.optim,
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=config.seed,
        report_to="wandb" if config.wandb else "none",
        bf16=True,
        packing=True,
        dataloader_num_workers=max(os.cpu_count() - 1, 1),
        dataloader_pin_memory=True,
        gradient_checkpointing=True,
        torch_compile=True,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=sft_config,
    )

    # Train only on responses
    from unsloth.chat_templates import train_on_responses_only

    logger.info("[SFT] Masking to train on response only!")
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

    trainer.train()
    logger.info("[SFT] Training completed.")

    # ── Save SFT adapter ────────────────────────────────────────────
    save_dir = config.sft_save_dir
    logger.info(f"[SFT] Saving model and tokenizer to {save_dir}")
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    !tar -czvf {save_dir}.tar.gz {save_dir}/

    # Free GPU before GRPO
    del model, tokenizer, trainer
    cleanup_gpu()
else:
    logger.info("[SFT] Skipping pre-SFT (--no-sft): GRPO-from-base comparison arm")
    # Free the model loaded for chat-template setup
    del model
    cleanup_gpu()

### Data Prep — GRPO

In [11]:
from datasets import load_dataset

dataset = load_dataset(config.grpo_data_path, split="train")
if config.sample_n > 0:
    dataset = dataset.select(range(min(config.sample_n, len(dataset))))
    logger.info(f"[data] Using subset of {len(dataset)} samples")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count'],
    num_rows: 93733
})

In [12]:
dataset[0]["problem"]

'## Task B-1.3.\n\nA ship traveling along a river has covered $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \\mathrm{~km}$ upstream and $21 \\mathrm{~km}$ downstream, or half an hour more than for traveling $15 \\mathrm{~km}$ upstream and $42 \\mathrm{~km}$ downstream, assuming that both the ship and the river move uniformly.\n\nDetermine the speed of the ship in still water and the speed of the river.'

In [13]:
dataset[0]["solution"]

'## Solution.\n\nLet $t$ be the time required for the boat to travel $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream, $v_{R}$ the speed of the river, and $v_{B}$ the speed of the boat. When the boat is traveling upstream, its speed is $v_{B}-v_{R}$, and when it is traveling downstream, its speed is $v_{B}+v_{R}$.\n\nSince $t=\\frac{s}{v}$, from the given data, we obtain the following system of equations:\n\n$\\left\\{\\begin{array}{l}t=\\frac{24}{v_{B}-v_{R}}+\\frac{28}{v_{B}+v_{R}} \\\\ t+0.5=\\frac{30}{v_{B}-v_{R}}+\\frac{21}{v_{B}+v_{R}} \\\\ t-0.5=\\frac{15}{v_{B}-v_{R}}+\\frac{42}{v_{B}+v_{R}}\\end{array}\\right.$\n\nBy introducing new variables $x=\\frac{3}{v_{B}-v_{R}}, y=\\frac{7}{v_{B}+v_{R}}$, the system transforms into:\n\n$\\left\\{\\begin{array}{l}t=8 x+4 y \\\\ t+0.5=10 x+3 y \\\\ t-0.5=5 x+6 y\\end{array}\\right.$\n\nSubstituting $t$ from the first equation into the remaining two, we get:\n\n$\\left\\{\\begin{array}{l}8 x+4 y+0.5=10 x+3 y \\\\ 8 x+4 y-0.5=5

In [14]:
def extract_hash_answer(text: str) -> str:
    """Extract the ground-truth answer from a solution string.

    For Open R1 dataset the solution text is already the answer,
    so we return it as-is.  For GSM8K-style ``####`` format, uncomment
    the splitting logic.
    """
    # if "####" not in text: return None
    # return text.split("####")[1].strip()
    return text


extract_hash_answer(dataset[0]["solution"])

'## Solution.\n\nLet $t$ be the time required for the boat to travel $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream, $v_{R}$ the speed of the river, and $v_{B}$ the speed of the boat. When the boat is traveling upstream, its speed is $v_{B}-v_{R}$, and when it is traveling downstream, its speed is $v_{B}+v_{R}$.\n\nSince $t=\\frac{s}{v}$, from the given data, we obtain the following system of equations:\n\n$\\left\\{\\begin{array}{l}t=\\frac{24}{v_{B}-v_{R}}+\\frac{28}{v_{B}+v_{R}} \\\\ t+0.5=\\frac{30}{v_{B}-v_{R}}+\\frac{21}{v_{B}+v_{R}} \\\\ t-0.5=\\frac{15}{v_{B}-v_{R}}+\\frac{42}{v_{B}+v_{R}}\\end{array}\\right.$\n\nBy introducing new variables $x=\\frac{3}{v_{B}-v_{R}}, y=\\frac{7}{v_{B}+v_{R}}$, the system transforms into:\n\n$\\left\\{\\begin{array}{l}t=8 x+4 y \\\\ t+0.5=10 x+3 y \\\\ t-0.5=5 x+6 y\\end{array}\\right.$\n\nSubstituting $t$ from the first equation into the remaining two, we get:\n\n$\\left\\{\\begin{array}{l}8 x+4 y+0.5=10 x+3 y \\\\ 8 x+4 y-0.5=5

In [15]:
dataset = dataset.map(
    lambda x: {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": x["problem"]},
        ],
        "answer": extract_hash_answer(x["solution"]),
    }
)
dataset[0]

Map:   0%|          | 0/93733 [00:00<?, ? examples/s]

{'problem': '## Task B-1.3.\n\nA ship traveling along a river has covered $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \\mathrm{~km}$ upstream and $21 \\mathrm{~km}$ downstream, or half an hour more than for traveling $15 \\mathrm{~km}$ upstream and $42 \\mathrm{~km}$ downstream, assuming that both the ship and the river move uniformly.\n\nDetermine the speed of the ship in still water and the speed of the river.',
 'solution': '## Solution.\n\nLet $t$ be the time required for the boat to travel $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream, $v_{R}$ the speed of the river, and $v_{B}$ the speed of the boat. When the boat is traveling upstream, its speed is $v_{B}-v_{R}$, and when it is traveling downstream, its speed is $v_{B}+v_{R}$.\n\nSince $t=\\frac{s}{v}$, from the given data, we obtain the following system of equations:\n\n$\\left\\{\\begin{array}{l}t=\\frac{24}{v_{B}-v_{R}}+\\fra

We create a regex format to match the reasoning sections and answers:

In [16]:
import re

solution_end_regex = (
    r"</SOLUTION>[\s]{0,}" + "(?:" + re.escape(tokenizer.eos_token) + ")?"
)

match_format = re.compile(
    rf"{REASONING_END}.*?"
    rf"{SOLUTION_START}(.+?){solution_end_regex}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL,
)
match_format

re.compile(r'<end_working_out>.*?<SOLUTION>(.+?)</SOLUTION>[\s]{0,}(?:<\|im_end\|>)?[\s]{0,}$',
re.MULTILINE|re.DOTALL|re.UNICODE)

In [17]:
match_format.findall(
    f"Let me think!<end_working_out><SOLUTION>\n2\n</SOLUTION>",
)
match_format.findall(
    f"<start_working_out>Let me think!<end_working_out><SOLUTION>  2  </SOLUTION>\n\n",
)

['  2  ']

In [18]:
def match_format_exactly(completions, **kwargs) -> list[float]:
    """Reward 3.0 if the completion matches the exact expected format."""
    scores = []
    for completion in completions:
        score = 0.0
        response = completion[0]["content"]
        if match_format.search(response) is not None:
            score += 3.0
        scores.append(score)
    return scores

In [19]:
def match_format_approximately(completions, **kwargs) -> list[float]:
    """Reward/penalize based on how many format tags are present."""
    scores = []
    for completion in completions:
        score = 0.0
        response = completion[0]["content"]
        score += 0.5 if response.count(REASONING_END) == 1 else -1.0
        score += 0.5 if response.count(SOLUTION_START) == 1 else -1.0
        score += 0.5 if response.count(SOLUTION_END) == 1 else -1.0
        scores.append(score)
    return scores

In [20]:
def check_answer(prompts, completions, answer, **kwargs) -> list[float]:
    """Reward based on correctness of the extracted answer vs ground truth."""
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1) if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0.0
        if guess is None:
            scores.append(-2.0)
            continue
        if guess == true_answer:
            score += 5.0
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            try:
                ratio = float(guess) / float(true_answer)
                if 0.9 <= ratio <= 1.1:
                    score += 2.0
                elif 0.8 <= ratio <= 1.2:
                    score += 1.5
                else:
                    score -= 2.5
            except Exception:
                score -= 4.5
        scores.append(score)
    return scores

In [21]:
match_numbers = re.compile(
    SOLUTION_START + r".*?[\s]{0,}([-]?[\d\.\,]{1,})", flags=re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  -0.234  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>17</SOLUTION>"))

['0.34']
['123,456']
['-0.234']
['17']


In [22]:
PRINTED_TIMES = 0
PRINT_EVERY_STEPS = 5


def check_numbers(prompts, completions, answer, **kwargs) -> list[float]:
    """Reward based on numeric match between extracted and true answer."""
    global PRINTED_TIMES
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1) if (guess := match_numbers.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            "*" * 20 + f"Question:\n{question}",
            f"\nAnswer:\n{answer[0]}",
            f"\nResponse:\n{responses[0]}",
            f"\nExtracted:\n{extracted_responses[0]}",
        )
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        try:
            true_answer = float(true_answer.strip())
            guess = float(guess.strip().replace(",", ""))
            scores.append(3.5 if guess == true_answer else -1.5)
        except Exception:
            scores.append(0.0)
            continue
    return scores

In [23]:
# ── Filter prompts by length (keep ≤ 90th percentile) ─────────────────
tokenized = dataset.map(
    lambda x: {
        "tokens": tokenizer.apply_chat_template(
            x["prompt"], add_generation_prompt=True, tokenize=True
        )
    },
    batched=True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L": len(x["tokens"])})

maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

Map:   0%|          | 0/93733 [00:00<?, ? examples/s]

You are given a problem.
Think about the problem and provide your working out.
Place it between <start_working_out> and <end_working_out>.
Then, provide your solution between <SOLUTION></SOLUTION><|im_end|>## Task B-1.3.

A ship traveling along a river has covered $24 \mathrm{~km}$ upstream and $28 \mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \mathrm{~km}$ upstream and $21 \mathrm{~km}$ downstream, or half an hour more than for traveling $15 \mathrm{~km}$ upstream and $42 \mathrm{~km}$ downstream, assuming that both the ship and the river move uniformly.

Determine the speed of the ship in still water and the speed of the river.<start_working_out>


Map:   0%|          | 0/93733 [00:00<?, ? examples/s]

Max Length =  205


### Train the model — GRPO

In [24]:
max_prompt_length = maximum_length + 1
max_completion_length = config.max_seq_length - max_prompt_length

print(f"Max prompt length: {max_prompt_length}")
print(f"Max completion length: {max_completion_length}")
from vllm import SamplingParams

vllm_sampling_params = SamplingParams(
    min_p=0.1,
    top_p=0.95,
    seed=config.seed,
    # Add both the EOS token and your custom solution end tag
    stop=[tokenizer.eos_token, SOLUTION_END],  
    include_stop_str_in_output=True,
)

from trl import GRPOConfig as TRLGRPOConfig, GRPOTrainer

training_args = TRLGRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=config.temperature,
    learning_rate=config.lr,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim=config.optim,
    logging_steps=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_generations=config.num_generations,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=config.max_steps,
    save_steps=config.max_steps // 2,
    report_to="wandb" if config.wandb else "none",
    output_dir=str(output_dir),
    seed=config.seed,
    run_name=RUN_NAME,
    # === Ablation knobs ===
    beta=config.beta,
    loss_type=config.loss_type,
    importance_sampling_level=config.importance_sampling_level,
    scale_rewards="group",
    torch_compile=True
)

Max prompt length: 206
Max completion length: 16178


In [25]:
# === WandB login ===
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "1"
os.environ["VLLM_ATTENTION_BACKEND"] = "FLASH_ATTN"

if config.wandb:
    try:
        import wandb
        if os.environ.get("WANDB_API_KEY"):
            wandb.login(key=os.environ["WANDB_API_KEY"])
        logger.info(
            f"[wandb] Logging to project={os.environ.get('WANDB_PROJECT')} "
            f"group={os.environ.get('WANDB_RUN_GROUP')} name={RUN_NAME} "
            f"mode={os.environ.get('WANDB_MODE', 'online')}"
        )
    except Exception as e:
        logger.warning(f"[wandb] Init failed ({e}) — continuing without wandb")


# === Custom callback: dump EVERY logged metric per run ===
from transformers import TrainerCallback


class MetricsDumpCallback(TrainerCallback):
    """Serializes all trainer metrics to ``outputs/<run_name>/metrics.json``.

    GRPOTrainer already logs (no duplication here): reward, reward_std,
    kl (only when beta != 0), entropy, completions/*, rewards/{func}/*,
    clip_ratio/*, loss, grad_norm, learning_rate, epoch.
    """

    def __init__(self, run_name: str, cfg: ExperimentConfig, out_dir: pathlib.Path):
        self.run_name = run_name
        self.config = cfg.to_dict()
        self.out_dir = pathlib.Path(out_dir)
        self.metrics_history: list[dict] = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        entry = {"step": state.global_step}
        entry.update({k: v for k, v in logs.items() if isinstance(v, (int, float))})
        self.metrics_history.append(entry)

    def on_train_end(self, args, state, control, **kwargs):
        dump = {
            "run_name": self.run_name,
            "config": self.config,
            "final_metrics": self.metrics_history[-1] if self.metrics_history else {},
            "metrics_history": self.metrics_history,
        }
        out_path = self.out_dir / "metrics.json"
        with open(out_path, "w") as f:
            json.dump(dump, f, indent=2, default=str)
        logger.info(f"[metrics] Saved to {out_path}")


metrics_callback = MetricsDumpCallback(RUN_NAME, config, output_dir)

# ── Save config before GRPO training ──────────────────────────────────
save_config_and_hyperparams(config, tokenizer, extra={"phase": "grpo"})

# ── Load model for GRPO ───────────────────────────────────────────────
# If config.adapter_path is set (or adapter_path arg is passed),
# the SFT adapter is loaded into the base model for Stage 2 warm-start.
# Otherwise a fresh LoRA adapter is created.
model, tokenizer = load_model(
    config=config,
    mode="train",
    stage="grpo",
    adapter_path="/kaggle/input/models/dzung271828/qwen3-1-7b-instruct/transformers/default/1/qwen_lora_sft",  # None → fresh LoRA; path → warm-start
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args=training_args,
    train_dataset=dataset,
    callbacks=[metrics_callback],
)
trainer.train()

# ── Post-training metrics fallback ────────────────────────────────────
if not (output_dir / "metrics.json").exists():
    fallback_metrics = {
        "run_name": RUN_NAME,
        "config": config.to_dict(),
        "final_metrics": trainer.state.log_history[-1] if trainer.state.log_history else {},
        "metrics_history": trainer.state.log_history,
    }
    with open(output_dir / "metrics.json", "w") as f:
        json.dump(fallback_metrics, f, indent=2, default=str)
    logger.info(f"[metrics] Fallback dump saved to {output_dir / 'metrics.json'}")

if config.wandb:
    try:
        import wandb
        wandb.finish()
    except Exception:
        pass

INFO 08-22 11:09:25 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading /kaggle/input/models/qwen-lm/qwen-3/transformers/1.7b-base/1 with actual GPU utilization = 59.54%
Unsloth: Your GPU has CUDA compute capability 12.0 with VRAM = 94.97 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 16384. Num Sequences = 128.
Unsloth: vLLM's KV Cache can use up to 53.12 GB. Also swap space = 6 GB.
Unsloth: Not an error, but `level` is not supported in vLLM.c

[W822 11:09:37.700659713 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


INFO 08-22 11:09:37 [topk_topp_sampler.py:51] Using FlashInfer for top-p & top-k sampling.
INFO 08-22 11:09:37 [gpu_model_runner.py:4481] Starting to load model /kaggle/input/models/qwen-lm/qwen-3/transformers/1.7b-base/1...
INFO 08-22 11:09:38 [cuda.py:317] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 08-22 11:09:38 [flash_attn.py:598] Using FlashAttention version 2


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


...

Unsloth: Just some info: will skip parsing ['norm1', 'norm2', 'post_attention_layernorm', 'ffn_norm', 'input_layernorm', 'k_norm', 'q_norm', 'attention_norm', 'layer_norm1', 'pre_feedforward_layernorm', 'post_feedforward_layernorm', 'post_layernorm', 'norm', 'layer_norm2']
Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['norm1', 'norm2', 'post_attention_layernorm', 'ffn_norm', 'input_layernorm', 'k_norm', 'q_norm', 'cross_attn_post_attention_layernorm', 'attention_norm', 'layer_norm1', 'pre_feedforward_layernorm', 'post_feedforward_layernorm', 'post_layernorm', 'norm', 'layer_norm2', 'cross_attn_input_layernorm']


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 84,442 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 34,865,152 of 1,755,440,128 (1.99% trained)


Unsloth: Enabled auto compiling
WARNING 08-22 11:11:35 [input_processor.py:141] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!
********************Question:
When $x=\frac{1}{5}$, the value of the expression $\frac{x^{2}-4}{x^{2}-2 x}$ is
(A) 0.4
(B) -0.52
(C) -5
(D) 10
(E) 11 
Answer:
Solution 1

We first simplify the expression:

$$
\frac{x^{2}-4}{x^{2}-2 x}=\frac{(x+2)(x-2)}{x(x-2)}=\frac{x+2}{x}=\frac{x}{x}+\frac{2}{x}=1+\frac{2}{x}
$$

(We can cancel the factor of $x-2$ since $x$ is not equal to 2.)

Substituting $x=\frac{1}{5}$, we get $1+\frac{2}{\left(\frac{1}{5}\right)}=1+10=11$.

## Solution 2

Substituting $x=\frac{1}{5}$,

$$
\frac{x^{2}-4}{x^{2}-2 x}=\frac{\frac{1}{25}-4}{\frac{1}{25}-\frac{2}{5}}=\frac{\frac{1}{25}-\fr

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_exactly / mean,rewards / match_format_exactly / std,rewards / match_format_approximately / mean,rewards / match_format_approximately / std,rewards / check_answer / mean,rewards / check_answer / std,rewards / check_numbers / mean,rewards / check_numbers / std
1,0.000000,0.000000,0.000000,2910.375000,1747.000000,5606.000000,1.000000,0.000000,0.000000,0.000000,0.000000,3.000000,0.000000,1.500000,0.000000,-4.500000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,4607.500000,2755.000000,8665.000000,1.000000,0.000000,0.000000,0.000000,0.000000,3.000000,0.000000,1.500000,0.000000,-4.500000,0.000000,0.000000,0.000000
3,-0.042400,-0.937500,2.651650,8830.000000,3838.000000,16178.000000,1.000000,0.000000,0.000000,0.000000,0.000000,2.625000,1.060660,0.937500,1.590990,-4.187500,0.883884,-0.312500,0.883884
4,0.000000,0.000000,0.000000,4991.250000,3189.000000,7693.000000,1.000000,0.000000,0.000000,0.000000,0.000000,3.000000,0.000000,1.500000,0.000000,-4.500000,0.000000,0.000000,0.000000
5,-0.000300,-0.562500,1.050085,8563.500000,6072.000000,14513.000000,1.000000,0.000000,0.000000,0.000000,0.000000,2.625000,1.060660,1.312500,0.530330,-4.187500,0.883884,-0.312500,0.883884
6,0.012500,-0.562500,1.590990,5991.000000,4474.000000,9734.000000,1.000000,0.000000,0.000000,0.000000,0.000000,2.625000,1.060660,1.312500,0.530330,-4.187500,0.883884,-0.312500,0.883884
7,0.000000,0.000000,0.000000,1262.125000,773.000000,1875.000000,1.000000,0.000000,0.000000,0.000000,0.000000,3.000000,0.000000,1.500000,0.000000,-4.500000,0.000000,0.000000,0.000000
8,-0.065000,-0.937500,2.651650,7454.375000,4488.000000,16178.000000,1.000000,0.000000,0.000000,0.000000,0.000000,2.625000,1.060660,0.937500,1.590990,-4.187500,0.883884,-0.312500,0.883884
9,-1.436800,-0.937500,2.651650,3077.875000,625.000000,16178.000000,1.000000,0.000000,0.000000,0.000000,0.000000,2.625000,1.060660,0.937500,1.590990,-4.187500,0.883884,-0.312500,0.883884
10,0.000000,0.000000,0.000000,5377.500000,2447.000000,8600.000000,1.000000,0.000000,0.000000,0.000000,0.000000,3.000000,0.000000,1.500000,0.000000,-4.500000,0.000000,0.000000,0.000000


********************Question:
B4. The infinite sequence of numbers

$$
0,1,2,2,1,-1,-2,-1,1,3, \ldots
$$

satisfies the following rule. For each quadruple of consecutive numbers $\ldots, a, b, c, d, \ldots$ in the sequence, it always holds that $d$ is equal to $c$ minus the smallest of the two numbers $a$ and $b$. Thus, the ninth number in the sequence is equal to $-1-(-2)=1$ and the tenth number is equal to $1-(-2)=3$. Calculate the 100th number in the sequence. 
Answer:
B4. With a little bit of arithmetic, we find more numbers in the sequence:

$$
\begin{aligned}
& 0, \quad 1, \quad 2, \quad 2, \quad 1, \quad -1, \quad -2, \quad -1, \quad 1, \quad 3, \quad 4, \quad 3, \quad 0, \quad -3, \quad -3, \\
& 0, \quad 3, \quad 6, \quad 6, \quad 3, \quad -3, \quad \ldots
\end{aligned}
$$

We see a clear pattern: after fifteen terms, the sequence repeats, but with all terms multiplied by three. To understand that the sequence indeed continues this pattern, consider the following. Each number i

### Inference

In [26]:
text = "What is the sqrt of 101?"

from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=8192,
    # Add both the EOS token and your custom solution end tag
    stop=[tokenizer.eos_token, SOLUTION_END],
    include_stop_str_in_output=True,
)

output = (
    model.fast_generate(
        [text],
        sampling_params=sampling_params,
        lora_request=None,
    )[0]
    .outputs[0]
    .text
)

output

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

" - Answers\nMath and Arithmetic\nAlgebra\nGeometry\nWhat is the sqrt of 101?\nWiki User\n∙ 2009-02-26 13:44:37\nStudy now\nBest Answer\nCopy\nsqrt(101) = 10.0499 (decimal approximation).\nWiki User\n∙ 2009-02-26 13:44:37\nThis answer is:\n🙏\n0\n🤨\n0\n😮\n0\nStudy guides\nAlgebra\n20 cards\nA polynomial of degree zero is a constant term\nThe grouping method of factoring can still be used when only some of the terms share a common factor A True B False\nThe sum or difference of p and q is the of the x-term in the trinomial\n3.85\n☆★☆★☆★☆★☆★\n262 Reviews\nWhat is the square root of 25101?\n158.3471\nWhat is the square root of 101?\nThe square root of 101 is about 10.05.\nWhat is the square root of 101 plus 4?\nsqrt(101) + 4 &asymp; 14.6666\nWhat is the square root of 101 added to 3?\nThe square root of 101 plus 3 is the same as the square root\nof 104 = 10.1980 ...\nsquare root of 101 plus 5?\nThe square root of 101 plus 5 is the same as the square root\nof 106 = 10.2956 ...\nWhat is the 

In [27]:
# Save the GRPO LoRA adapter
model.save_lora(str(output_dir / GRPO_LORA_SAVE_NAME))

In [28]:
# Verify LoRA weights are non-trivial
from safetensors import safe_open

tensors = {}
with safe_open(
    str(output_dir / GRPO_LORA_SAVE_NAME / "adapter_model.safetensors"), framework="pt"
) as f:
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert n_zeros.item() != tensor.numel()

logger.info("[verify] LoRA adapter weights are non-trivial (not all zeros).")

In [29]:
# Test with the trained GRPO LoRA
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
)
from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=8192,
    # Add both the EOS token and your custom solution end tag
    stop=[tokenizer.eos_token, SOLUTION_END],
    include_stop_str_in_output=True,
)

output = (
    model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=model.load_lora(str(output_dir / GRPO_LORA_SAVE_NAME)),
    )[0]
    .outputs[0]
    .text
)

output

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"เตือน\n\n戀\n\n<start_working_out>Okay, let's see. I need to find the square root of 101. Hmm. I remember that the square root of a number is a value that, when multiplied by itself, gives the original number. So, I need to find a number which, when multiplied by itself, equals 101. Not sure if 101 is something I've heard of before. Wait, 10 squared is 100, so that's already close. What's 11 squared? Let me check. 11 times 11 is 121. Oh, so 11 squared is 121, which is higher than 101. That means the square root of 101 is somewhere between 10 and 11. Wait, no, wait. Wait, 10 squared is 100, so the square root of 100 is 10. Then 11 squared is 121, so the square root of 121 is 11. So between 10 and 11. But 101 is just 1 less than 121. So it should be around 10.05 or something? Let me try that. 10.05 times 10.05. Let's calculate that. 10 times 10 is 100, 0.05 times 10 is 0.5, and then 0.05 times 10.05. Wait, that might be more complicated. Wait, 10.05 squared. Let's break it down. That's (

### Saving to float16 for VLLM

In [30]:
# Merge to 16bit
if False:
    model.save_pretrained_merged(
        "qwen_finetune_16bit",
        tokenizer,
        save_method="merged_16bit",
    )
if False:
    model.push_to_hub_merged(
        "HF_USERNAME/qwen_finetune_16bit",
        tokenizer,
        save_method="merged_16bit",
        token="YOUR_HF_TOKEN",
    )

# Merge to 4bit
if False:
    model.save_pretrained_merged(
        "qwen_finetune_4bit",
        tokenizer,
        save_method="merged_4bit",
    )
if False:
    model.push_to_hub_merged(
        "HF_USERNAME/qwen_finetune_4bit",
        tokenizer,
        save_method="merged_4bit",
        token="YOUR_HF_TOKEN",
    )

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False:
    model.push_to_hub("HF_USERNAME/qwen_lora", token="YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token="YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion

In [31]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf(
        "qwen_finetune",
        tokenizer,
    )
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", tokenizer, token="YOUR_HF_TOKEN"
    )

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method="f16")
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune",
        tokenizer,
        quantization_method="f16",
        token="YOUR_HF_TOKEN",
    )

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method="q4_k_m")
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune",
        tokenizer,
        quantization_method="q4_k_m",
        token="YOUR_HF_TOKEN",
    )

# Save to multiple GGUF options
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune",
        tokenizer,
        quantization_method=["q4_k_m", "q8_0", "q5_k_m"],
        token="YOUR_HF_TOKEN",
    )